# Deep Learning Assignment 2

In [2]:
from pathlib import Path
import h5py
import numpy as np

from scipy.signal import resample

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [3]:
data_root = Path('..') / 'data' / 'raw' / 'Final Project data'
h5_files = sorted(data_root.rglob('*.h5'))

if not h5_files:
    raise FileNotFoundError(f'No .h5 files found under {data_root}')

def get_dataset_name(file_name_with_dir: Path) -> str:
    file_name_without_dir = file_name_with_dir.as_posix().split('/')[-1]
    temp = file_name_without_dir.split('_')[:-1]
    dataset_name = "_".join(temp)
    return dataset_name

filename_path = h5_files[0]
print(f'Loading: {filename_path}')

with h5py.File(filename_path, 'r') as f:
    dataset_name = get_dataset_name(filename_path)
    obj = f.get(dataset_name)
    if obj is None:
        raise ValueError(f"Dataset '{dataset_name}' not found in {filename_path}")
    if not isinstance(obj, h5py.Dataset):
        raise ValueError(f"Expected h5py.Dataset, got {type(obj)}")
    dataset: h5py.Dataset = obj
    matrix: np.ndarray = dataset[()]
    print(type(matrix))
    print(matrix.shape)

Loading: ../data/raw/Final Project data/Cross/test1/rest_162935_1.h5
<class 'numpy.ndarray'>
(248, 35624)


In [4]:
# Define labels and map to PyTorch integer targets
# Mapping the raw filename prefixes to the classification targets (rest, motor, math, memory)
label_mapping = {
    "rest": 0,                
    "task_motor": 1,          
    "task_story_math": 2,     
    "task_working_memory": 3  
}

def get_label(filepath: Path) -> int:
    """Extracts the task from the filename and returns the integer label."""
    filename = filepath.stem
    for task, label in label_mapping.items():
        if filename.startswith(task):
            return label
    raise ValueError(f"Unknown task in file: {filename}")

# Quick test on the first file
sample_file = h5_files[0]
print(f"Sample file: {sample_file.name} -> Label ID: {get_label(sample_file)}")

Sample file: rest_162935_1.h5 -> Label ID: 0


In [5]:
# Preprocess the time series data and save to disk

processed_root = Path('..') / 'data' / 'processed' / 'Final Project data'

# Downsampling factor (e.g., reducing 2034 Hz to ~203 Hz by a factor of 10)
# changes the 35624 time steps to a much more manageable 3562
DOWNSAMPLE_FACTOR = 10 

def preprocess_matrix(matrix: np.ndarray) -> np.ndarray:
    """Downsamples and normalizes the MEG data time-wise."""
    
    target_length = matrix.shape[1] // DOWNSAMPLE_FACTOR
    # resample applies a Fourier method, which is excellent for signals
    matrix_resampled = resample(matrix, target_length, axis=1)
    
    # Calculate mean and standard deviation along the time axis (axis=1)
    mean = np.mean(matrix_resampled, axis=1, keepdims=True)
    std = np.std(matrix_resampled, axis=1, keepdims=True)
    
    # Prevent division by zero just in case a channel is perfectly flat
    std[std == 0] = 1e-8
    
    matrix_normalized = (matrix_resampled - mean) / std
    return matrix_normalized

print(f"Starting preprocessing... Target directory: {processed_root}")

for i, file_path in enumerate(h5_files):
    # Reconstruct the folder structure for the processed data
    relative_path = file_path.relative_to(data_root)
    target_path = processed_root / relative_path
    
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    if target_path.exists():
        continue
        
    with h5py.File(file_path, 'r') as f:
        dataset_name = get_dataset_name(file_path)
        matrix = f.get(dataset_name)[()]
        
    processed_matrix = preprocess_matrix(matrix)
    
    with h5py.File(target_path, 'w') as f:
        f.create_dataset(dataset_name, data=processed_matrix)
        
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(h5_files)} files...")

print("Preprocessing complete! Data is ready for the DataLoader.")

sample_processed_path = processed_root / h5_files[0].relative_to(data_root)
with h5py.File(sample_processed_path, 'r') as f:
    dataset_name = get_dataset_name(sample_processed_path)
    sample_matrix = f.get(dataset_name)[()]
    print(f"\nOriginal shape: {matrix.shape}")
    print(f"Processed shape: {sample_matrix.shape}")

Starting preprocessing... Target directory: ../data/processed/Final Project data
Preprocessing complete! Data is ready for the DataLoader.

Original shape: (248, 35624)
Processed shape: (248, 3562)


In [6]:
# Prepare the intra-subject dataset

intra_train_dir = processed_root / 'Intra' / 'train'
intra_test_dir = processed_root / 'Intra' / 'test'

intra_train_files = sorted(list(intra_train_dir.glob('*.h5')))
intra_test_files = sorted(list(intra_test_dir.glob('*.h5')))

class MEGDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        
        with h5py.File(file_path, 'r') as f:
            dataset_name = get_dataset_name(file_path)
            matrix = f.get(dataset_name)[()]
            
        x = torch.tensor(matrix, dtype=torch.float32)
        y = get_label(file_path)
        
        return x, y

intra_train_dataset = MEGDataset(intra_train_files)
intra_test_dataset = MEGDataset(intra_test_files)

# A batch size of 8 or 16 is ideal here since the intra-subject file count is small
BATCH_SIZE = 16

intra_train_loader = DataLoader(intra_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
intra_test_loader = DataLoader(intra_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ Intra-Subject Train dataset created with {len(intra_train_dataset)} samples.")
print(f"✓ Intra-Subject Test dataset created with {len(intra_test_dataset)} samples.")

if len(intra_train_loader) > 0:
    images, labels = next(iter(intra_train_loader))
    print(f"\n--- DataLoader Sanity Check ---")
    print(f"Batch X shape (batch_size, channels, time): {images.shape}")
    print(f"Batch y shape (labels): {labels.shape}")

✓ Intra-Subject Train dataset created with 32 samples.
✓ Intra-Subject Test dataset created with 8 samples.

--- DataLoader Sanity Check ---
Batch X shape (batch_size, channels, time): torch.Size([16, 248, 3562])
Batch y shape (labels): torch.Size([16])


In [7]:
# Prepare the cross-subject dataset

cross_train_dir = processed_root / 'Cross' / 'train'
cross_test1_dir = processed_root / 'Cross' / 'test1'
cross_test2_dir = processed_root / 'Cross' / 'test2'
cross_test3_dir = processed_root / 'Cross' / 'test3'

cross_train_files = sorted(list(cross_train_dir.glob('*.h5')))
cross_test1_files = sorted(list(cross_test1_dir.glob('*.h5')))
cross_test2_files = sorted(list(cross_test2_dir.glob('*.h5')))
cross_test3_files = sorted(list(cross_test3_dir.glob('*.h5')))

cross_train_dataset = MEGDataset(cross_train_files)
cross_test1_dataset = MEGDataset(cross_test1_files)
cross_test2_dataset = MEGDataset(cross_test2_files)
cross_test3_dataset = MEGDataset(cross_test3_files)

# A batch size of 16 works nicely here. 
CROSS_BATCH_SIZE = 16

cross_train_loader = DataLoader(cross_train_dataset, batch_size=CROSS_BATCH_SIZE, shuffle=True)
cross_test1_loader = DataLoader(cross_test1_dataset, batch_size=CROSS_BATCH_SIZE, shuffle=False)
cross_test2_loader = DataLoader(cross_test2_dataset, batch_size=CROSS_BATCH_SIZE, shuffle=False)
cross_test3_loader = DataLoader(cross_test3_dataset, batch_size=CROSS_BATCH_SIZE, shuffle=False)

print(f"✓ Cross-Subject Train dataset created with {len(cross_train_dataset)} samples.")
print(f"✓ Cross-Subject Test 1 dataset created with {len(cross_test1_dataset)} samples.")
print(f"✓ Cross-Subject Test 2 dataset created with {len(cross_test2_dataset)} samples.")
print(f"✓ Cross-Subject Test 3 dataset created with {len(cross_test3_dataset)} samples.")

if len(cross_train_loader) > 0:
    images, labels = next(iter(cross_train_loader))
    print(f"\n--- Cross DataLoader Sanity Check ---")
    print(f"Batch X shape (batch_size, channels, time): {images.shape}")
    print(f"Batch y shape (labels): {labels.shape}")

✓ Cross-Subject Train dataset created with 64 samples.
✓ Cross-Subject Test 1 dataset created with 16 samples.
✓ Cross-Subject Test 2 dataset created with 16 samples.
✓ Cross-Subject Test 3 dataset created with 16 samples.

--- Cross DataLoader Sanity Check ---
Batch X shape (batch_size, channels, time): torch.Size([16, 248, 3562])
Batch y shape (labels): torch.Size([16])


In [8]:
# Define the model architecture

class MEGClassifier(nn.Module):
    def __init__(self, num_classes=4, num_channels=248):
        super(MEGClassifier, self).__init__()
        
        # Block 1: Input shape (Batch, 248, 3562)
        self.conv1 = nn.Conv1d(in_channels=num_channels, out_channels=64, kernel_size=15, stride=2, padding=7)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=4, stride=4) # downsamples sequence length
        
        # Block 2: Shape (Batch, 64, 445)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7, stride=1, padding=3)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # Block 3: Shape (Batch, 128, 111)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.AdaptiveAvgPool1d(1) # collapses the time dimension completely down to 1
        
        # Fully Connected Classifier Stage
        self.dropout = nn.Dropout(p=0.5) # prevent overfitting on cross-subject splits
        self.fc = nn.Linear(256, num_classes)
        
    def forward(self, x):
        # forward pass through conv Layers
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        
        # flatten temporal dimension: output becomes (Batch, 256)
        x = torch.flatten(x, 1)
        
        # output classification logits
        x = self.dropout(x)
        x = self.fc(x)
        return x

model_dummy = MEGClassifier()
print(model_dummy)

dummy_input = torch.randn(2, 248, 3562) # Batch of 2, 248 sensors, 3562 time steps
dummy_output = model_dummy(dummy_input)
print(f"\nSanity Check pass:")
print(f"Input shape: {dummy_input.shape}")
print(f"Output logits shape: {dummy_output.shape} (Expected: [2, 4])")

MEGClassifier(
  (conv1): Conv1d(248, 64, kernel_size=(15,), stride=(2,), padding=(7,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu1): ReLU()
  (pool1): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu2): ReLU()
  (pool2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu3): ReLU()
  (pool3): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)

Sanity Check pass:
Input shape: torch.Size([2, 248, 3562])
Output logits shape: torch.Size

In [9]:
# Train and evaluate the intra-subject model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

intra_model = MEGClassifier(num_classes=4, num_channels=248).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(intra_model.parameters(), lr=0.001, weight_decay=1e-4)

NUM_EPOCHS = 20

print("\n--- Starting Intra-Subject Training ---")
for epoch in range(NUM_EPOCHS):
    intra_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_x, batch_y in intra_train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # zero out gradients from the previous step
        optimizer.zero_grad()
        
        outputs = intra_model(batch_x)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / total_train
    epoch_acc = (correct_train / total_train) * 100
    
    intra_model.eval()
    correct_test = 0
    total_test = 0
    
    # disable gradient tracking during evaluation to save memory and computations
    with torch.no_grad():
        for batch_x, batch_y in intra_test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = intra_model(batch_x)
            _, predicted = torch.max(outputs, 1)
            total_test += batch_y.size(0)
            correct_test += (predicted == batch_y).sum().item()
            
    test_acc = (correct_test / total_test) * 100
    
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] -> "
          f"Train Loss: {epoch_loss:.4f} | "
          f"Train Acc: {epoch_acc:.2f}% | "
          f"Test Acc: {test_acc:.2f}%")

print("\nIntra-Subject Training Complete!")

Using device: cuda

--- Starting Intra-Subject Training ---
Epoch [01/20] -> Train Loss: 1.3446 | Train Acc: 40.62% | Test Acc: 50.00%
Epoch [02/20] -> Train Loss: 1.0312 | Train Acc: 65.62% | Test Acc: 50.00%
Epoch [03/20] -> Train Loss: 0.9063 | Train Acc: 65.62% | Test Acc: 50.00%
Epoch [04/20] -> Train Loss: 0.7821 | Train Acc: 75.00% | Test Acc: 50.00%
Epoch [05/20] -> Train Loss: 0.6688 | Train Acc: 81.25% | Test Acc: 50.00%
Epoch [06/20] -> Train Loss: 0.5749 | Train Acc: 93.75% | Test Acc: 50.00%
Epoch [07/20] -> Train Loss: 0.4928 | Train Acc: 96.88% | Test Acc: 62.50%
Epoch [08/20] -> Train Loss: 0.4501 | Train Acc: 96.88% | Test Acc: 62.50%
Epoch [09/20] -> Train Loss: 0.3874 | Train Acc: 93.75% | Test Acc: 62.50%
Epoch [10/20] -> Train Loss: 0.2908 | Train Acc: 100.00% | Test Acc: 50.00%
Epoch [11/20] -> Train Loss: 0.2434 | Train Acc: 100.00% | Test Acc: 37.50%
Epoch [12/20] -> Train Loss: 0.1599 | Train Acc: 100.00% | Test Acc: 62.50%
Epoch [13/20] -> Train Loss: 0.1457 |

In [10]:
# Train and evaluate the cross-subject model

cross_model = MEGClassifier(num_classes=4, num_channels=248).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cross_model.parameters(), lr=0.001, weight_decay=1e-4)

CROSS_EPOCHS = 20

print("--- Starting Cross-Subject Training ---")
for epoch in range(CROSS_EPOCHS):
    # --- TRAINING PHASE ---
    cross_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_x, batch_y in cross_train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = cross_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / total_train
    epoch_acc = (correct_train / total_train) * 100
    
    cross_model.eval()
    
    def evaluate_loader(loader):
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = cross_model(batch_x)
                _, predicted = torch.max(outputs, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        return (correct / total) * 100 if total > 0 else 0.0

    test1_acc = evaluate_loader(cross_test1_loader)
    test2_acc = evaluate_loader(cross_test2_loader)
    test3_acc = evaluate_loader(cross_test3_loader)
    
    print(f"Epoch [{epoch+1:02d}/{CROSS_EPOCHS}] -> "
          f"Train Loss: {epoch_loss:.4f} | "
          f"Train Acc: {epoch_acc:.2f}% || "
          f"Test1 Acc: {test1_acc:.2f}% | "
          f"Test2 Acc: {test2_acc:.2f}% | "
          f"Test3 Acc: {test3_acc:.2f}%")

print("\nCross-Subject Training Complete!")

--- Starting Cross-Subject Training ---
Epoch [01/20] -> Train Loss: 1.3019 | Train Acc: 39.06% || Test1 Acc: 25.00% | Test2 Acc: 25.00% | Test3 Acc: 25.00%
Epoch [02/20] -> Train Loss: 0.9487 | Train Acc: 60.94% || Test1 Acc: 25.00% | Test2 Acc: 50.00% | Test3 Acc: 37.50%
Epoch [03/20] -> Train Loss: 0.5989 | Train Acc: 85.94% || Test1 Acc: 25.00% | Test2 Acc: 50.00% | Test3 Acc: 43.75%
Epoch [04/20] -> Train Loss: 0.5150 | Train Acc: 84.38% || Test1 Acc: 25.00% | Test2 Acc: 50.00% | Test3 Acc: 50.00%
Epoch [05/20] -> Train Loss: 0.4051 | Train Acc: 96.88% || Test1 Acc: 31.25% | Test2 Acc: 68.75% | Test3 Acc: 43.75%
Epoch [06/20] -> Train Loss: 0.3561 | Train Acc: 89.06% || Test1 Acc: 62.50% | Test2 Acc: 56.25% | Test3 Acc: 56.25%
Epoch [07/20] -> Train Loss: 0.2637 | Train Acc: 100.00% || Test1 Acc: 62.50% | Test2 Acc: 50.00% | Test3 Acc: 68.75%
Epoch [08/20] -> Train Loss: 0.2581 | Train Acc: 100.00% || Test1 Acc: 75.00% | Test2 Acc: 50.00% | Test3 Acc: 68.75%
Epoch [09/20] -> Train

In [11]:
# Compare results and implement a stronger alternative approach (1D ResNet)

# alternative architecture: Residual 1D CNN Block
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        # shortcut connection to match dimensions if channels or resolution changes
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

# full Improved Classifier
class MEGResNetClassifier(nn.Module):
    def __init__(self, num_classes=4, num_channels=248):
        super(MEGResNetClassifier, self).__init__()
        
        # initial projection layer downsampling the sequence length
        self.init_conv = nn.Conv1d(num_channels, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.init_bn = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # residual stages
        self.layer1 = ResidualBlock1D(64, 64, stride=1)
        self.layer2 = ResidualBlock1D(64, 128, stride=2)
        self.layer3 = ResidualBlock1D(128, 256, stride=2)
        
        # global pooling and classification
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool(self.relu(self.init_bn(self.init_conv(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

resnet_model = MEGResNetClassifier(num_classes=4, num_channels=248).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet_model.parameters(), lr=0.0005, weight_decay=1e-3) # stronger weight decay for regularization

ALT_EPOCHS = 15
print("--- Training Alternative Baseline (ResNet 1D) on Cross-Subject Split ---")
for epoch in range(ALT_EPOCHS):
    resnet_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_x, batch_y in cross_train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = resnet_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / total_train
    epoch_acc = (correct_train / total_train) * 100
    
    resnet_model.eval()
    def evaluate_resnet(loader):
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = resnet_model(batch_x)
                _, predicted = torch.max(outputs, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        return (correct / total) * 100 if total > 0 else 0.0

    t1_acc = evaluate_resnet(cross_test1_loader)
    t2_acc = evaluate_resnet(cross_test2_loader)
    t3_acc = evaluate_resnet(cross_test3_loader)
    
    print(f"Epoch [{epoch+1:02d}/{ALT_EPOCHS}] -> Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% || "
          f"Test1: {t1_acc:.2f}% | Test2: {t2_acc:.2f}% | Test3: {t3_acc:.2f}%")

print("\nAlternative Model Training Complete!")

--- Training Alternative Baseline (ResNet 1D) on Cross-Subject Split ---
Epoch [01/15] -> Train Loss: 1.3530 | Train Acc: 28.12% || Test1: 37.50% | Test2: 37.50% | Test3: 50.00%
Epoch [02/15] -> Train Loss: 0.9722 | Train Acc: 53.12% || Test1: 50.00% | Test2: 62.50% | Test3: 50.00%
Epoch [03/15] -> Train Loss: 0.6629 | Train Acc: 79.69% || Test1: 56.25% | Test2: 68.75% | Test3: 43.75%
Epoch [04/15] -> Train Loss: 0.4779 | Train Acc: 82.81% || Test1: 50.00% | Test2: 62.50% | Test3: 50.00%
Epoch [05/15] -> Train Loss: 0.3404 | Train Acc: 90.62% || Test1: 43.75% | Test2: 56.25% | Test3: 50.00%
Epoch [06/15] -> Train Loss: 0.1881 | Train Acc: 96.88% || Test1: 43.75% | Test2: 81.25% | Test3: 50.00%
Epoch [07/15] -> Train Loss: 0.1644 | Train Acc: 96.88% || Test1: 37.50% | Test2: 68.75% | Test3: 56.25%
Epoch [08/15] -> Train Loss: 0.1165 | Train Acc: 100.00% || Test1: 50.00% | Test2: 62.50% | Test3: 62.50%
Epoch [09/15] -> Train Loss: 0.0594 | Train Acc: 100.00% || Test1: 68.75% | Test2: 62.

In [13]:
# Train the Alternative Baseline (ResNet 1D) on Intra-Subject Split

intra_resnet_model = MEGResNetClassifier(num_classes=4, num_channels=248).to(device)
optimizer_intra_res = torch.optim.Adam(intra_resnet_model.parameters(), lr=0.0005, weight_decay=1e-3)

print("\n--- Training Alternative Baseline (ResNet 1D) on Intra-Subject Split ---")
for epoch in range(ALT_EPOCHS):
    intra_resnet_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_x, batch_y in intra_train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer_intra_res.zero_grad()
        outputs = intra_resnet_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer_intra_res.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / total_train
    epoch_acc = (correct_train / total_train) * 100
    
    # Quick evaluation for the epoch printout
    intra_resnet_model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for batch_x, batch_y in intra_test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = intra_resnet_model(batch_x)
            _, predicted = torch.max(outputs, 1)
            total_test += batch_y.size(0)
            correct_test += (predicted == batch_y).sum().item()
            
    test_acc = (correct_test / total_test) * 100 if total_test > 0 else 0.0
    
    print(f"Epoch [{epoch+1:02d}/{ALT_EPOCHS}] -> Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print("\nIntra-Subject ResNet Training Complete!")


--- Training Alternative Baseline (ResNet 1D) on Intra-Subject Split ---
Epoch [01/15] -> Train Loss: 1.4043 | Train Acc: 34.38% | Test Acc: 37.50%
Epoch [02/15] -> Train Loss: 1.1241 | Train Acc: 43.75% | Test Acc: 25.00%
Epoch [03/15] -> Train Loss: 0.7997 | Train Acc: 68.75% | Test Acc: 25.00%
Epoch [04/15] -> Train Loss: 0.6846 | Train Acc: 75.00% | Test Acc: 37.50%
Epoch [05/15] -> Train Loss: 0.5070 | Train Acc: 93.75% | Test Acc: 37.50%
Epoch [06/15] -> Train Loss: 0.3885 | Train Acc: 96.88% | Test Acc: 50.00%
Epoch [07/15] -> Train Loss: 0.3362 | Train Acc: 96.88% | Test Acc: 50.00%
Epoch [08/15] -> Train Loss: 0.1886 | Train Acc: 100.00% | Test Acc: 50.00%
Epoch [09/15] -> Train Loss: 0.1113 | Train Acc: 100.00% | Test Acc: 50.00%
Epoch [10/15] -> Train Loss: 0.1385 | Train Acc: 100.00% | Test Acc: 50.00%
Epoch [11/15] -> Train Loss: 0.0560 | Train Acc: 100.00% | Test Acc: 50.00%
Epoch [12/15] -> Train Loss: 0.0438 | Train Acc: 100.00% | Test Acc: 62.50%
Epoch [13/15] -> Trai

In [14]:
# Final Evaluation and Metric Comparison

from sklearn.metrics import classification_report, confusion_matrix, f1_score

def evaluate_and_report(model, dataloader, dataset_name):
    """Evaluates the model on a dataloader and prints a breakdown of performance."""
    model.eval()
    all_preds = []
    all_labels = []
    
    # map back to target names
    target_names = ["rest", "motor", "math", "memory"]
    
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.numpy())
            
    # calculate overall accuracy
    correct = sum(1 for p, l in zip(all_preds, all_labels) if p == l)
    total = len(all_labels)
    accuracy = (correct / total) * 100
    
    # calculate macro F1 score (unweighted mean of F1 for all classes)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    print(f"\n==================================================")
    print(f" Evaluation Report: {dataset_name}")
    print(f" Overall Accuracy: {accuracy:.2f}% | Macro F1 Score: {f1:.4f}")
    print(f"==================================================")
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    
    return accuracy, f1

print("Running final evaluation on Intra-Subject Test set (Baseline CNN)...")
intra_final_acc, intra_final_f1 = evaluate_and_report(intra_model, intra_test_loader, "Intra-Subject Test Set (Baseline CNN)")

print("\nRunning final evaluation on Intra-Subject Test set (1D ResNet)...")
intra_resnet_acc, intra_resnet_f1 = evaluate_and_report(intra_resnet_model, intra_test_loader, "Intra-Subject Test Set (1D ResNet)")

print("\nRunning final evaluation on Baseline 1D CNN Cross-Subject Test sets...")
c_base_t1_acc, c_base_t1_f1 = evaluate_and_report(cross_model, cross_test1_loader, "Baseline Cross-Subject - Unseen Subject 1")
c_base_t2_acc, c_base_t2_f1 = evaluate_and_report(cross_model, cross_test2_loader, "Baseline Cross-Subject - Unseen Subject 2")
c_base_t3_acc, c_base_t3_f1 = evaluate_and_report(cross_model, cross_test3_loader, "Baseline Cross-Subject - Unseen Subject 3")

baseline_cross_avg_acc = (c_base_t1_acc + c_base_t2_acc + c_base_t3_acc) / 3
baseline_cross_avg_f1 = (c_base_t1_f1 + c_base_t2_f1 + c_base_t3_f1) / 3

print("\nRunning final evaluation on Alternative 1D ResNet Cross-Subject Test sets...")
c_res_t1_acc, c_res_t1_f1 = evaluate_and_report(resnet_model, cross_test1_loader, "1D ResNet Cross-Subject - Unseen Subject 1")
c_res_t2_acc, c_res_t2_f1 = evaluate_and_report(resnet_model, cross_test2_loader, "1D ResNet Cross-Subject - Unseen Subject 2")
c_res_t3_acc, c_res_t3_f1 = evaluate_and_report(resnet_model, cross_test3_loader, "1D ResNet Cross-Subject - Unseen Subject 3")

resnet_cross_avg_acc = (c_res_t1_acc + c_res_t2_acc + c_res_t3_acc) / 3
resnet_cross_avg_f1 = (c_res_t1_f1 + c_res_t2_f1 + c_res_t3_f1) / 3

# --- Displaying the Final Summary Table ---
print("\n" + "="*80)
print(f"{'SETTING / MODEL':<40} | {'AVG TEST ACCURACY':<18} | {'MACRO F1 SCORE':<15}")
print("="*80)
print(f"{'Intra-Subject (Baseline 1D CNN)':<40} | {intra_final_acc:>16.2f}% | {intra_final_f1:>14.4f}")
print(f"{'Intra-Subject (Alternative 1D ResNet)':<40} | {intra_resnet_acc:>16.2f}% | {intra_resnet_f1:>14.4f}")
print("-" * 80)
print(f"{'Cross-Subject (Baseline 1D CNN)':<40} | {baseline_cross_avg_acc:>16.2f}% | {baseline_cross_avg_f1:>14.4f}")
print(f"{'Cross-Subject (Alternative 1D ResNet)':<40} | {resnet_cross_avg_acc:>16.2f}% | {resnet_cross_avg_f1:>14.4f}")
print("="*80)

Running final evaluation on Intra-Subject Test set (Baseline CNN)...

 Evaluation Report: Intra-Subject Test Set (Baseline CNN)
 Overall Accuracy: 75.00% | Macro F1 Score: 0.7417
              precision    recall  f1-score   support

        rest       1.00      1.00      1.00         2
       motor       1.00      0.50      0.67         2
        math       0.50      0.50      0.50         2
      memory       0.67      1.00      0.80         2

    accuracy                           0.75         8
   macro avg       0.79      0.75      0.74         8
weighted avg       0.79      0.75      0.74         8

Confusion Matrix:
[[2 0 0 0]
 [0 1 1 0]
 [0 0 1 1]
 [0 0 0 2]]

Running final evaluation on Intra-Subject Test set (1D ResNet)...

 Evaluation Report: Intra-Subject Test Set (1D ResNet)
 Overall Accuracy: 50.00% | Macro F1 Score: 0.4500
              precision    recall  f1-score   support

        rest       1.00      1.00      1.00         2
       motor       0.33      0.50      0